In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

num_customers = 10000
customer_ids = range(100000001, 100000001 + num_customers)

genders = np.random.choice(['Male', 'Female'], size=num_customers)

def generate_dob():
    older_dobs = pd.to_datetime(np.random.choice(
        pd.date_range('1935-01-01', '1944-12-31'),
        size=100
    ))
    remaining_dobs = pd.to_datetime(np.random.choice(
        pd.date_range('1945-01-01', '2007-12-31'),
        size=num_customers - 100
    ))

    return np.concatenate([older_dobs, remaining_dobs])

dobs = generate_dob()
ages = 2025 - pd.DatetimeIndex(dobs).year

# Địa điểm và vùng miền
locations = [
    'Hà Nội', 'Hải Phòng', 'Quảng Ninh', 'Bắc Ninh',
    'Thanh Hóa', 'Đà Nẵng', 'Khánh Hòa',
    'TP. Hồ Chí Minh', 'Đồng Nai', 'Bình Thuận',
    'Bà Rịa – Vũng Tàu', 'TP. Cần Thơ', 'Hưng Yên',
    'Nghệ An', 'Bình Định', 'Long An', 'Bình Dương'
]
location_choices = np.random.choice(locations, size=num_customers)

def get_region(loc):
    if loc in ['Hà Nội', 'Hải Phòng', 'Quảng Ninh', 'Bắc Ninh', 'Hưng Yên']:
        return 'Miền Bắc'
    elif loc in ['Thanh Hóa', 'Đà Nẵng', 'Khánh Hòa', 'Nghệ An', 'Bình Định']:
        return 'Miền Trung'
    else:
        return 'Miền Nam'

location_regions = [get_region(loc) for loc in location_choices]

# Nghề nghiệp
occupations = []
for age in ages:
    if age <= 22:
        occupations.append('Sinh viên')
    else:
        occupations.append(np.random.choice([
            'Kinh doanh', 'Giáo viên', 'Nhân viên văn phòng', 'Công nhân',
            'Lao động phổ thông', 'Công chức', 'Nội trợ', 'Tự do', 'Khác'
        ]))

# Phân khúc khách hàng
customer_segments = np.random.choice(
    ['Mass', 'Mass Aspirational', 'Premium', 'Super Premium'],
    size=num_customers,
    p=[0.4, 0.3, 0.2, 0.1]
)

# Tạo DataFrame
df = pd.DataFrame({
    'customer_id': customer_ids,
    'gender': genders,
    'dob': dobs,
    'age': ages,
    'location': location_choices,
    'location_region': location_regions,
    'occupation': occupations,
    'customer_segment': customer_segments
})


df.to_csv('MIS_dim_customer.csv', index=False, encoding='utf-8-sig')

In [ ]:
df_customer = pd.DataFrame({'customer_id': range(100000001, 100010001)})
df_transaction = pd.DataFrame({'customer_id': np.random.choice(df_customer['customer_id'], size=100000)})

transaction_counts = df_transaction['customer_id'].value_counts().to_dict()

num_customers = df_customer.shape[0]
start_date = datetime(2024, 6, 30)

# Random dữ liệu
last_logins = [start_date + timedelta(days=random.randint(0, 365)) for _ in range(num_customers)]
login_7d = np.random.randint(0, 11, size=num_customers)
login_30d = login_7d + np.random.randint(0, 11, size=num_customers)
mobile_ratio = np.round(np.random.uniform(0.44, 1.0, size=num_customers), 2)
support_usage = np.random.randint(0, 10, size=num_customers)

# Tính phản hồi
positive_feedback = []
negative_feedback = []
for cid in df_customer['customer_id']:
    total_txn = transaction_counts.get(cid, 0)
    if total_txn > 1:
        # positive luôn lớn hơn negative
        pos = random.randint((total_txn // 2) + 1, total_txn)
        neg = total_txn - pos
    elif total_txn == 1:
        pos = 1
        neg = 0
    else:
        pos = 0
        neg = 0
    positive_feedback.append(pos)
    negative_feedback.append(neg)

# Tạo bảng Service Usage
df_service_usage = pd.DataFrame({
    'customer_id': df_customer['customer_id'],
    'last_login_mobile_app': last_logins,
    'login_count_7d': login_7d,
    'login_count_30d': login_30d,
    'mobile_transaction_ratio': mobile_ratio,
    'support_usage_count': support_usage,
    'positive_feedback_count': positive_feedback,
    'negative_feedback_count': negative_feedback
})

df_service_usage.to_csv('fact_service_usage_limited.csv', index=False, encoding='utf-8-sig')


In [ ]:
num_records = 25000
campaign_ids = list(range(3000001, 3000021))

# Tạo mapping: mỗi campaign có 2–5 content_id riêng
campaign_content_map = {}
for cid in campaign_ids:
    num_content = random.randint(2, 5)
    campaign_content_map[cid] = [f"CT_{cid}_{i}" for i in range(1, num_content + 1)]

# Sinh dữ liệu cho 25000 dòng
customer_ids = np.random.choice(df_customer['customer_id'], size=num_records, replace=True)
assigned_campaigns = np.random.choice(campaign_ids, size=num_records, replace=True)
assigned_contents = [random.choice(campaign_content_map[cid]) for cid in assigned_campaigns]

# Channel logic
channels = np.random.choice(
    ['SMS', 'Email', 'Chatbot', 'Telesales'],
    size=num_records,
    p=[0.4, 0.3, 0.2, 0.1]
)

# Thời gian gửi
start_date = datetime(2024, 6, 30)
timestamps = [start_date + timedelta(days=random.randint(0, 365)) for _ in range(num_records)]

# Phản hồi khách hàng
outcomes = np.random.choice(
    ['No action', 'Opened', 'Clicked', 'Replied', 'Converted'],
    size=num_records,
    p=[0.5, 0.25, 0.15, 0.07, 0.03]
)

df_marketing = pd.DataFrame({
    'campaign_id': assigned_campaigns,
    'customer_id': customer_ids,
    'channel': channels,
    'timestamp': timestamps,
    'content_id': assigned_contents,
    'outcome': outcomes
})

df_marketing.to_csv('fact_marketing_crm.csv', index=False, encoding='utf-8-sig')

# Hiển thị 5 dòng đầu để kiểm tra logic
print(df_marketing.head())

   campaign_id  customer_id    channel  timestamp    content_id    outcome
0      3000016    100009577        SMS 2025-04-29  CT_3000016_3  No action
1      3000010    100003235      Email 2025-02-25  CT_3000010_3     Opened
2      3000018    100008926      Email 2025-06-09  CT_3000018_2     Opened
3      3000005    100000848  Telesales 2025-05-31  CT_3000005_1  No action
4      3000005    100006449        SMS 2024-07-30  CT_3000005_2  No action


In [ ]:
# Danh sách chi nhánh
branches = [
    'Hà Nội', 'Hải Phòng', 'Quảng Ninh', 'Bắc Ninh',
    'Thanh Hóa', 'Đà Nẵng', 'Khánh Hòa',
    'TP. Hồ Chí Minh', 'Đồng Nai', 'Bình Thuận',
    'Bà Rịa – Vũng Tàu', 'TP. Cần Thơ', 'Hưng Yên',
    'Nghệ An', 'Bình Định', 'Long An', 'Bình Dương'
]

# Cấu hình số lượng nhân viên mỗi phòng ban/chi nhánh
dept_config = {
    'Giao dịch': 4,
    'Tín dụng': 4,
    'Dịch vụ khách hàng': 4,
    'Vận hành kỹ thuật': 2
}

# Tổng nhân sự
num_employees = len(branches) * sum(dept_config.values())
employee_ids = [f"EMP{str(i).zfill(4)}" for i in range(1, num_employees + 1)]

# Tạo dữ liệu
records = []
emp_index = 0

for branch in branches:
    for dept, count in dept_config.items():
        if dept == 'Vận hành kỹ thuật':
            monthly_kpi = 0
        else:
            monthly_kpi = np.random.randint(60, 84)

        tx_per_day_list = []
        max_tx = min(8, monthly_kpi // 8) if monthly_kpi > 0 else 0
        twelve_indices = []
        if count >= 4 and max_tx >= 12:
            twelve_indices = np.random.choice(range(count), size=4, replace=False)
        for i in range(count):
            if i in twelve_indices:
                tx_per_day = 12
            else:
                tx_per_day = np.random.randint(1, max_tx + 1) if max_tx >= 1 else 0
            tx_per_day_list.append(tx_per_day)

        for i in range(count):
            emp_id = employee_ids[emp_index]
            emp_index += 1

            tx_per_day = tx_per_day_list[i]

            if dept == 'Giao dịch':
                avg_time = round(np.random.uniform(3, 8), 2)
                satisfaction = round(np.random.uniform(0.6, 0.9), 2)
            elif dept == 'Tín dụng':
                avg_time = round(np.random.uniform(15, 30), 2)
                satisfaction = round(np.random.uniform(0.6, 0.9), 2)
            elif dept == 'Dịch vụ khách hàng':
                avg_time = round(np.random.uniform(5, 15), 2)
                satisfaction = round(np.random.uniform(0.6, 0.9), 2)
            else:  # Vận hành kỹ thuật
                tx_per_day = None
                avg_time = round(np.random.uniform(10, 20), 2)
                satisfaction = None

            error_rate = round(np.random.uniform(0.0, 4.0), 2)

            records.append({
                'employee_id': emp_id,
                'branch': branch,
                'department': dept,
                'transactions_per_day': tx_per_day,
                'avg_handling_time_min': avg_time,
                'customer_satisfaction': satisfaction,
                'monthly_kpi': monthly_kpi,
                'error_rate_percent': error_rate
            })

df_employee_ops = pd.DataFrame(records)

df_employee_ops.to_csv('dim_employee_ops_cleaned.csv', index=False, encoding='utf-8-sig')

print(df_employee_ops.head(10))

  employee_id  branch          department  transactions_per_day  \
0     EMP0001  Hà Nội           Giao dịch                   5.0   
1     EMP0002  Hà Nội           Giao dịch                   8.0   
2     EMP0003  Hà Nội           Giao dịch                   6.0   
3     EMP0004  Hà Nội           Giao dịch                   7.0   
4     EMP0005  Hà Nội            Tín dụng                   5.0   
5     EMP0006  Hà Nội            Tín dụng                   6.0   
6     EMP0007  Hà Nội            Tín dụng                   4.0   
7     EMP0008  Hà Nội            Tín dụng                   4.0   
8     EMP0009  Hà Nội  Dịch vụ khách hàng                   7.0   
9     EMP0010  Hà Nội  Dịch vụ khách hàng                   2.0   

   avg_handling_time_min  customer_satisfaction  monthly_kpi  \
0                   5.35                   0.72           75   
1                   6.21                   0.75           75   
2                   7.93                   0.73           75   
3     

In [ ]:
df_transaction = pd.read_csv('fact_transactions.csv')

# Giới hạn số cảnh báo phù hợp ngân hàng vừa
num_alerts = 3000
alert_ids = [f'ALERT{str(i).zfill(5)}' for i in range(1, num_alerts + 1)]

# Lấy ngẫu nhiên các transaction_id đã có
transaction_ids = np.random.choice(df_transaction['transaction_id'], size=num_alerts, replace=False)

# Lấy customer_id tương ứng từ bảng transaction
transaction_lookup = df_transaction.set_index('transaction_id')['customer_id'].to_dict()
customer_ids = [transaction_lookup[txn_id] for txn_id in transaction_ids]

# Tạo timestamp dựa trên thời gian giao dịch hoặc giả lập
start_date = datetime(2024, 6, 30)
timestamps = [start_date + timedelta(days=random.randint(0, 365)) for _ in range(num_alerts)]

# Điểm rủi ro AI dự đoán: 20% > 0.3, còn lại < 0.3
num_high = int(num_alerts * 0.2)
num_low = num_alerts - num_high

# Sinh điểm thấp (<0.3)
risk_scores_low = np.round(np.random.beta(a=2, b=8, size=num_low) * 0.3, 4)
# Sinh điểm cao (>0.3)
risk_scores_high = np.round(0.3 + np.random.beta(a=2, b=2, size=num_high) * 0.7, 4)

# Kết hợp và xáo trộn
risk_scores = np.concatenate([risk_scores_low, risk_scores_high])
np.random.shuffle(risk_scores)

# Kết quả xác nhận: 12% thực sự gian lận
results = np.random.choice(['false positive', 'confirmed fraud'], size=num_alerts, p=[0.88, 0.12])

df_fraud = pd.DataFrame({
    'alert_id': alert_ids,
    'customer_id': customer_ids,
    'transaction_id': transaction_ids,
    'timestamp': timestamps,
    'risk_score': risk_scores,
    'result': results
})

df_fraud.to_csv('fact_fraud_alerts.csv', index=False, encoding='utf-8-sig')